In [ ]:


# in prompts folder, prompt_service.py file 

from langchain.prompts import PromptTemplate

class PromptService:

    @staticmethod
    def getAnalyseSystemImpactsPrompt() -> PromptTemplate:
        prompt = """
          Using formal software architecture methodology, you will analyse the following systems that's written in JSON format:

          {SystemImpactedDataJSON}

          As a first step, extract all system and their descriptions. Use only the information from the above project brief. The systems impacts will have a clear view of what systems are impacted in this project and a detail description of what they do. 

          Return the response in the following JSON format. Return "Empty" if no information could be extracted from the project brief. Do not return any explanation. Your response must be a valid JSON document only.:

          {{
          "systems" : {{"system name": "system name",
                        "System Description": "system description"}}
                        
          }}
          """

        return PromptTemplate.from_template(prompt)

    @staticmethod
    def getAnalyseEventsPrompt() -> PromptTemplate:
        prompt = """
      Using formal software architecture methodology, you will analyse the following project brief that's written in JSON format:
        
      {ProjectBriefDataJSON}
      
      As a first step, extract all manual or automated events from the project brief. Use only the information from the above project brief. Events are triggers than a system or user role can initiate when the solution is deployed into the insurance firm.

      Return the response in the following JSON format. Return "Empty" if no information could be extracted from the project brief. Do not return any explanation. Your response must be a valid JSON document only.:

      {{
      "manual_events" : {{"event name": "event description"}},
      "automated_events" :  {{"event name": "event description"}}
      }}
      """

        return PromptTemplate.from_template(prompt)


# in utils folder, create_prompt.py file

from prompts import prompt_service
import json, logging

class PromptCreator:
    # this can be based on a template in the future - e.g. workflow / no code style
    # Dispatcher function lookup. Set on init
    process_dispatcher = None
    document_template_dispatcher = None
    data = None # the data used to generate the prompts


    def __init__(self):
        self.prompt_service = prompt_service.PromptService()
        self.process_dispatcher = {
            'roles': self.analyse_roles,
            'events': self.analyse_events,
            'activities': self.analyse_activities,
            'systems_impacted': self.analyse_systems_impacted,
            "service_catalog": self.determine_services,
            "capability_L1_impact_analysis" : self.determine_level1_impacts,
            "capability_L2_impact_analysis" : self.determine_level_two_impacts, # purposely named different to level 1 to minimise accidental errors as functions will look similar


        }

    def execute_step (self, process_step: str, *args):
        
        execute_step = self.process_dispatcher.get(process_step, self.default_process_step)
        prompt = execute_step(*args)

        return prompt
        
    def analyse_systems_impacted (self):

        prompt_template = self.prompt_service.getAnalyseSystemImpactsPrompt()

        if 'systems_impacted' in self.data:
            systems_impacted = self.data['systems_impacted']  # extract out the events data
            del self.data['systems_impacted']
        else:
            print("ERROR - cannot call analyse activities. Missing events_data.")
            return

        prompt = prompt_template.format(SystemImpactedDataJSON=systems_impacted)

        return prompt

    def analyse_events (self):
        
        prompt_template = self.prompt_service.getAnalyseEventsPrompt()

        prompt = prompt_template.format(ProjectBriefDataJSON=json.dumps(self.data))

        return prompt

    def set_data (self, variables: dict, clean_unused=False):
        self.data = variables




# in main.py file

from utils import create_prompt 


promtCreator = create_prompt.PromptCreator()

subset_data = promtCreator.create_subset_data(data)
ProjectBriefDataJSON = subset_data.copy() # ProjectBriefDataJSON shows as keyword in github code colors
promtCreator.set_data(subset_data)  # add in the required data for this instance of prompt creator


prompt_system_impact = promtCreator.execute_step("systems_impacted")

# we have the relevant prompt now 


In [21]:
from pydantic import BaseModel, HttpUrl, ValidationError
from pydantic_settings import BaseSettings

class UrlValidator(BaseSettings):
    url: HttpUrl

def validate(url: str):
    try:
        UrlValidator(url=url)
    except ValidationError:
        print("Not a valid url")
    else:
        print("Valid url")

In [22]:
validate("https://payg-kar-d-r1-oai.openai.azure.com")

Valid url


In [4]:
import configparser

In [12]:
con = configparser.ConfigParser()
con.read("azure_config.ini")

['azure_config.ini']

In [13]:
con??

Type:        ConfigParser
String form: <configparser.ConfigParser object at 0x10b2023d0>
Length:      3
File:        /Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/configparser.py
Source:     
class ConfigParser(RawConfigParser):
    """ConfigParser implementing interpolation."""

    _DEFAULT_INTERPOLATION = BasicInterpolation()

    def set(self, section, option, value=None):
        """Set an option.  Extends RawConfigParser.set by validating type and
        interpolation syntax on the value."""
        self._validate_value_types(option=option, value=value)
        super().set(section, option, value)

    def add_section(self, section):
        """Create a new section in the configuration.  Extends
        RawConfigParser.add_section by validating if the section name is
        a string."""
        self._validate_value_types(section=section)
        super().add_section(section)

    def _read_defaults(self, defaults):
        """

In [15]:
con['azure_deployment_config']["azure_endpoint"]

'https://payg-kar-d-r1-oai.openai.azure.com'

In [16]:
print(con['azure_deployment_config']["azure_endpoint"])
print(type(con['azure_deployment_config']["azure_endpoint"]))

https://payg-kar-d-r1-oai.openai.azure.com
<class 'str'>


In [17]:
k = con['azure_deployment_config']["azure_endpoint"]

In [18]:
type(k)

str

In [23]:
validate(k)

Valid url


In [26]:

from pydantic import BaseModel, HttpUrl, ValidationError
from pydantic_settings import SettingsConfigDict, BaseSettings
import configparser
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AzureDeploymentConfig(BaseSettings):
    model_config = SettingsConfigDict(env_prefix='vk_')
    azure_endpoint: HttpUrl
    deployment_name: str
    openai_api_version: str
    token_scope: str

    def __init__(self):
        # Initialize base settings
        self.model_config = SettingsConfigDict(env_prefix='vk_')
        
        # Load configuration
        azure_config = configparser.ConfigParser()
        azure_config.read('azure_config.ini')

        try:
            azure_endpoint = azure_config['azure_deployment_config']['azure_endpoint'].strip().strip('"')
            deployment_name = azure_config['azure_deployment_config']['deployment_name'].strip()
            openai_api_version = azure_config['azure_deployment_config']['openai_api_version'].strip()
            token_scope = azure_config['azure_deployment_config']['token_scope'].strip()

            # Log the values read from config file
            logger.info(f"Read from config: {azure_endpoint}, {deployment_name}, {openai_api_version}, {token_scope}")

            # Assign values after reading from config
            self.azure_endpoint = azure_endpoint
            self.deployment_name = deployment_name
            self.openai_api_version = openai_api_version
            self.token_scope = token_scope
        except KeyError as e:
            logger.error(f"Missing required configuration value: {e}")
            raise ValueError(f"Missing required configuration value: {e}")
        except ValidationError as e:
            logger.error(f"Validation error: {e}")
            raise

        # Log the loaded configuration
        logger.info(f"Loaded Azure configuration: {self.dict()}")

# from config import AzureDeploymentConfig

try:
    config = AzureDeploymentConfig()
    print(f"Configuration loaded successfully: {config}")
except Exception as e:
    print(f"Error loading configuration: {e}")

Error loading configuration: "AzureDeploymentConfig" object has no field "model_config"


Error loading configuration: "AzureDeploymentConfig" object has no field "model_config"
